# Inria 3DGS (10k) on Colab T4

This notebook trains the original Inria 3D Gaussian Splatting model for 10,000 iterations using a COLMAP scene from Google Drive and records:

- PSNR, SSIM, LPIPS
- Training time
- Peak GPU memory
- Number of Gaussians
- PLY file size

Use **Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
from pathlib import Path
import json
import csv
import os
import re
import subprocess
import threading
import time

SCENE_NAME = "gerrard-hall"
MAX_ITERATIONS = 10_000

# Expected dataset layout in Drive:
# MyDrive/worldmodeling/scenes/gerrard-hall/images
# MyDrive/worldmodeling/scenes/gerrard-hall/sparse
DRIVE_ROOT = Path("/content/drive/MyDrive/worldmodeling")
SCENE_DIR = DRIVE_ROOT / "scenes" / SCENE_NAME

WORK_ROOT = Path("/content/worldmodeling_inria")
REPO_DIR = WORK_ROOT / "gaussian-splatting"
RUNS_ROOT = DRIVE_ROOT / "sota_runs"
MODEL_DIR = RUNS_ROOT / "inria_3dgs" / SCENE_NAME
METRICS_DIR = RUNS_ROOT / "metrics"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Scene:", SCENE_NAME)
print("Iterations:", MAX_ITERATIONS)
print("Scene dir:", SCENE_DIR)
print("Model dir:", MODEL_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

required = [SCENE_DIR / "images", SCENE_DIR / "sparse"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required dataset paths in Drive:\n" + "\n".join(missing)
    )

num_images = len(list((SCENE_DIR / "images").glob("*")))
print(f"Drive mounted. Found {num_images} image files.")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select a T4 GPU runtime and rerun.")

gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name)
print(f"Total VRAM: {gpu.total_memory / 1e9:.2f} GB")
if "T4" not in gpu.name:
    print("Warning: runtime is not a T4. Metrics will still run, but comparison scope assumes T4.")

In [ ]:
def run_cmd(cmd, cwd=None, check=True, capture=True):
    print(f"\\n$ {cmd}")
    result = subprocess.run(
        cmd,
        cwd=cwd,
        shell=True,
        text=True,
        capture_output=capture,
    )
    if capture:
        if result.stdout:
            print(result.stdout[-3000:])
        if result.stderr:
            print(result.stderr[-3000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")
    return result

def build_gpu_monitor(interval_sec=2.0):
    state = {"max_mem_mb": 0}
    stop_event = threading.Event()

    def _poll():
        while not stop_event.is_set():
            try:
                out = subprocess.check_output(
                    "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                    shell=True,
                    text=True,
                ).strip()
                mem = int(out.splitlines()[0])
                state["max_mem_mb"] = max(state["max_mem_mb"], mem)
            except Exception:
                pass
            time.sleep(interval_sec)

    thread = threading.Thread(target=_poll, daemon=True)
    return state, stop_event, thread

def parse_metrics_text(text):
    metrics = {"PSNR": None, "SSIM": None, "LPIPS": None}
    patterns = {
        "PSNR": r"PSNR[^0-9]*([0-9]+(?:\\.[0-9]+)?)",
        "SSIM": r"SSIM[^0-9]*([0-9]+(?:\\.[0-9]+)?)",
        "LPIPS": r"LPIPS[^0-9]*([0-9]+(?:\\.[0-9]+)?)",
    }
    for key, pattern in patterns.items():
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            metrics[key] = float(m.group(1))
    return metrics

def count_vertices_from_ply(ply_path):
    with open(ply_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("element vertex"):
                return int(line.split()[-1])
            if line == "end_header":
                break
    return None

In [ ]:
# Clone and install Inria 3DGS dependencies
run_cmd("pip install -q ninja plyfile lpips tqdm")

if not REPO_DIR.exists():
    run_cmd(f"git clone https://github.com/graphdeco-inria/gaussian-splatting.git {REPO_DIR}", capture=False)
else:
    print("Repository already exists, reusing local checkout.")

run_cmd("git pull", cwd=REPO_DIR)
run_cmd("pip install -q -r requirements.txt", cwd=REPO_DIR)
run_cmd("pip install -q ./submodules/diff-gaussian-rasterization ./submodules/simple-knn", cwd=REPO_DIR)

print("Inria 3DGS install complete.")

In [ ]:
# Train for 10k iterations with evaluation split enabled
train_cmd = (
    f"python train.py -s '{SCENE_DIR}' -m '{MODEL_DIR}' "
    f"--iterations {MAX_ITERATIONS} --eval"
)

gpu_state, gpu_stop, gpu_thread = build_gpu_monitor()
gpu_thread.start()

t0 = time.time()
train_result = run_cmd(train_cmd, cwd=REPO_DIR)
training_time_sec = time.time() - t0

gpu_stop.set()
gpu_thread.join(timeout=5)

peak_gpu_gb = gpu_state["max_mem_mb"] / 1024
print(f"Training time: {training_time_sec/60:.2f} minutes")
print(f"Peak GPU memory: {peak_gpu_gb:.2f} GB")

In [ ]:
# Render test views and compute image quality metrics
render_cmd = f"python render.py -m '{MODEL_DIR}' --iteration {MAX_ITERATIONS} --skip_train"
metrics_cmd = f"python metrics.py -m '{MODEL_DIR}'"

run_cmd(render_cmd, cwd=REPO_DIR)
metrics_result = run_cmd(metrics_cmd, cwd=REPO_DIR)
metrics_from_stdout = parse_metrics_text((metrics_result.stdout or "") + "\n" + (metrics_result.stderr or ""))
print("Parsed metrics from console:", metrics_from_stdout)

In [ ]:
# Gather final metrics and artifacts
point_cloud_ply = MODEL_DIR / "point_cloud" / f"iteration_{MAX_ITERATIONS}" / "point_cloud.ply"
if not point_cloud_ply.exists():
    raise FileNotFoundError(f"Expected PLY not found: {point_cloud_ply}")

num_gaussians = count_vertices_from_ply(point_cloud_ply)
ply_size_mb = point_cloud_ply.stat().st_size / 1e6

metrics_json_candidates = [
    MODEL_DIR / "results.json",
    MODEL_DIR / "metrics.json",
]
file_metrics = {}
for p in metrics_json_candidates:
    if p.exists():
        with open(p, "r") as f:
            loaded = json.load(f)
        if isinstance(loaded, dict):
            file_metrics = loaded
        break

psnr = metrics_from_stdout["PSNR"]
ssim = metrics_from_stdout["SSIM"]
lpips = metrics_from_stdout["LPIPS"]

summary = {
    "model": "Inria_3DGS",
    "scene": SCENE_NAME,
    "iterations": MAX_ITERATIONS,
    "PSNR": psnr,
    "SSIM": ssim,
    "LPIPS": lpips,
    "training_time_sec": round(training_time_sec, 3),
    "training_time_min": round(training_time_sec / 60.0, 3),
    "peak_gpu_gb": round(peak_gpu_gb, 3),
    "num_gaussians": num_gaussians,
    "ply_file_size_mb": round(ply_size_mb, 3),
    "ply_path": str(point_cloud_ply),
    "raw_metrics_file": file_metrics,
}

json_path = METRICS_DIR / f"inria_3dgs_{SCENE_NAME}_10k.json"
csv_path = METRICS_DIR / "sota_metrics_10k.csv"

with open(json_path, "w") as f:
    json.dump(summary, f, indent=2)

csv_headers = [
    "model", "scene", "iterations", "PSNR", "SSIM", "LPIPS",
    "training_time_sec", "training_time_min", "peak_gpu_gb",
    "num_gaussians", "ply_file_size_mb", "ply_path"
]
write_header = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_headers)
    if write_header:
        writer.writeheader()
    writer.writerow({k: summary.get(k) for k in csv_headers})

print("Saved JSON:", json_path)
print("Appended CSV:", csv_path)
print(json.dumps(summary, indent=2))

## Notes

- The CSV is shared across models so you can run the Mip-Splatting notebook next and compare rows directly.
- If `PSNR/SSIM/LPIPS` show as `null`, inspect the output of `metrics.py` and adjust regex parsing to match the current upstream print format.